# rl19_grid

> Phase 19 (`specs/multiagent/phase19/`): training the guard role with RL (contest credit). `RL19Config` (contest slots, the bonus half-life, the all-sharing penalty, the varied-threat mix, per-agent records and the answer halt) with `configHash19` (Phase 17-18 hashes unchanged); the arms and runs (`ARMS19`, `SEEDS19`, `RUNS19`, `BASE19`, `rl19Config`); the speed check (`SPEED19`), the squad-size pilot (`PILOT19_PLAN`, `pilotChoice19`) and the main plan's skeleton (`RL19_PLAN`, its decisions P19-1 to P19-7 as the user decided them, unfrozen); the training world (`worldFor19`, `needWorld19`: squads joined by bridges, P19-1, from Gate C1's record, `rl19_bridge.GATE_C1`, with `bridge1` on every arm; squads without cross-squad subtasks only if Gate C1 fails); `projectHours19` and `evalAdapter19` (the checkpoint before a halting streak, the answer halt included). The evaluation grid is in `rl19_suite`.

In [ ]:
#| default_exp rl19_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## The configuration

In [ ]:
#| export
import hashlib
import json
from dataclasses import dataclass

from explRL.coop_grid import RL_PLAN
from explRL.rl17_grid import BASE17, RL17_PLAN, RL17Config, runFields17
from explRL.rl18_grid import FIX18
from explRL.rl19_bridge import GATE_C1, bridgeWorld19

LIFETIMES = (15, 30)                    # P19-7: subtask lifetimes in turns (the board's default life is 30)
VARIED, FIXED4 = ((0, 4, 6), LIFETIMES), ((4,), LIFETIMES)   # liars of 16 and lifetimes drawn per training board


@dataclass
class RL19Config(RL17Config):
    contestSlots: str = None    # None (credit reward), 'guards' (arms 2, 3) or 'all' (arm 4): who earns contest credit
    halfLife: float = 4.0       # P19-2: the prevention bonus halves every halfLife turns after the lie
    sharePenalty: float = 0.0   # P19-3: taken from every honest slot when honest answers < a third of WARM_ANSWERS
    threat: tuple = None        # VARIED or FIXED4: each board's liar count (of 16) and lifetime, drawn per step
    records: bool = False       # arm 4 (P19-5): each agent's own record of payoffs over a sequence of 6 boards
    haltAnswers: int = None     # FR-A8: halt after this many steps in a row with answers below a third of WARM_ANSWERS


RL19_DEFAULTS = dict(contestSlots=None, halfLife=4.0, sharePenalty=0.0, threat=None, records=False, haltAnswers=None)


def runFields19(cfg):
    "runFields17 without Phase 19's fields at their defaults (an RL17Config: runFields17 itself)."
    return {k: v for k, v in runFields17(cfg).items() if k not in RL19_DEFAULTS or v != RL19_DEFAULTS[k]}


def configHash19(cfg):
    "configHash17's rule over runFields19: an RL19Config at the new fields' defaults has its RL17Config's hash."
    d = {k: v for k, v in runFields19(cfg).items() if k not in ('out', 'device', 'ckptEvery', 'microBatch', 'steps', 'gradCkpt')}
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]

## Arms, runs and the plans

In [ ]:
#| export
ARMS19 = dict(
    a19credit=dict(scheme='credit', contestSlots=None, threat=VARIED, guards=4),         # 1: credit only, varied threat
    a19contest=dict(scheme='contest', contestSlots='guards', threat=VARIED, guards=4),   # 2: contest credit, varied threat
    a19fixed=dict(scheme='contest', contestSlots='guards', threat=FIXED4, guards=4),     # 3: contest credit, 4 liars of 16
    a19open=dict(scheme='contest', contestSlots='all', threat=VARIED, guards=0, records=None))   # 4: None: set by P19-5
SEEDS19 = dict(a19credit=8, a19contest=8, a19fixed=8, a19open=5)
RUNS19 = [(a, s) for s in range(1, 9) for a in ARMS19 if s <= SEEDS19[a]]   # 29 runs, interleaved
STEPS19 = 30
BASE19 = dict(**BASE17, **FIX18, steps=STEPS19, world=(('colony', True), ('r', 2)), N=16, mixN=((16, 1.0),), haltAnswers=3)
WARM_ANSWERS = None    # the speed check's tables: {squad size: {guards: [((16, liars, lifetime), honest answers per board), ...]}}
DECISIONS19 = dict(    # the user's decisions (2026-11-08); P19-6 is confirmed after the speed check
    P19_1=dict(value='bridges', proposal='squads joined by bridges, as the plan intends: A1 waits for C1 (Phase 20) to be built and Gate C1 to run; squads without cross-squad subtasks only if Gate C1 fails (user, 2026-11-08)', confirmed=True),
    P19_2=dict(value=4.0, proposal='the prevention bonus halves every 4 turns after the lie', confirmed=True),
    P19_3=dict(value=0.5, proposal='0.5 from every honest slot when honest answers < a third of the warm start\'s', confirmed=True),
    P19_4=dict(value=(2, 8), proposal='held-out liar shares 2 and 8 of 16 at evaluation', confirmed=True),
    P19_5=dict(value=None, proposal='A2 runs first (user, 2026-11-08); arm 4 uses records if Gate A2 passes, none if it fails; set at the freeze', confirmed=True),
    P19_6=dict(value=295, proposal='about 295 A100 hours; decided after the speed check reports A100 hours and compute units (user, 2026-11-08)', confirmed=False),
    P19_7=dict(value=LIFETIMES, proposal='subtask lifetimes 15 and 30 turns', confirmed=True),
    power=dict(value=1.5, proposal='the primary tests are powered for 1.5 SD with 8 seeds per arm; a 1-SD effect would likely read as not shown (user, 2026-11-08)', confirmed=True))
WORLDS19 = ('bridges', 'squads')       # P19-1: squads joined by bridges; squads without cross-squad subtasks only if Gate C1 fails
BRIDGES_READY = True                   # Phase 20's bridged rollouts exist; A1's wiring is rl19_bridge (T5a, 2026-11-12)


def worldFor19(c1):
    "The world from Gate C1's record ({passed, inertFired, userAcceptsBridges}, or bool passed): 'bridges' if it passed and bridges are kept (P19-8)."
    if c1 is None:
        raise ValueError('Gate C1 is not recorded: A1 waits for C1 (P19-1)')
    c1 = c1 if isinstance(c1, dict) else dict(passed=bool(c1), inertFired=False)
    return 'bridges' if c1['passed'] and (not c1['inertFired'] or c1.get('userAcceptsBridges')) else 'squads'


def needWorld19(world):
    "Refuse a world that cannot run yet: 'bridges' until Phase 20's bridged rollouts exist; anything outside WORLDS19."
    if world not in WORLDS19:
        raise ValueError(f'world must be one of {WORLDS19}, got {world!r}')
    if world == 'bridges' and not BRIDGES_READY:
        raise NotImplementedError('squads joined by bridges need Phase 20 (C1, specs/multiagent/phase20/): its bridged rollouts '
                                  'do not exist yet; A1 waits for them (P19-1; squads without bridges only if Gate C1 fails)')
    return world


SPEED19 = dict(
    version=1, frozen=None, arm='a19contest', seed=1, steps=3, squads=(4, 8), maxLen=4096, window=3072, dir='speed_p19',
    world=DECISIONS19['P19_1']['value'], warmSeeds=list(range(960, 965)), warmGuards=(4, 0),
    measures='seconds per step (step 1 excluded), the longest prompt plus reply, a bridge agent\'s reported apart (it reads its own '
    'squad\'s board and its pair\'s mirror lines, P20-3; the 3,072 window is refused if either exceeds it), invalid replies, WARM_ANSWERS (warm-start rollouts on '
    'every training board type, per squad size, with 4 guards and without guards), Colab compute units per A100 hour (read on '
    'the account)', note='no penalty and no answer halt: WARM_ANSWERS is unknown')
PILOT19_PLAN = dict(
    version=1, frozen=None, arm='a19contest', steps=10, seed=1, squads=(4, 8), world=DECISIONS19['P19_1']['value'],
    dir='runs_p19pilot', evalDir='eval_p19pilot',
    heldOut=list(range(950, 955)), selection='honest welfare (welfareHonest) at 4 liars of 16 minus the warm start\'s on the '
    'held-out boards per adapter; the larger gain wins, squads of 4 win ties', choice=None)
RL19_PLAN = dict(
    version=1, frozen=None, userConfirmed=None, decisions=DECISIONS19, squads=None, world=DECISIONS19['P19_1']['value'], records=None,
    arms=ARMS19, seeds=SEEDS19, runs=[list(r) for r in RUNS19], steps=STEPS19, recipe=dict(RL17_PLAN['recipe']), maxLen=3072,
    turnFraction=None, warmAnswers=None, projectedHours=None, power=None, gates=dict(C1=GATE_C1, A2=None),
    common='RL_PLAN v2 + BASE17 (FIX17; wrong answers cost 3 in training) + FIX18 + BASE19; trainSeed = seed - 1',
    evaluation='rl19_suite.contestSuite: 16 agents in the chosen squads of the training world, liars 0, 2, 4, 6, 8 of 16, lifetimes '
               '15 and 30, seeds 900-909, 40 turns, the default (individual) reward scheme in the prompts (sequences: arm 4\'s, contest '
               'credit open to all); the warm start without guards (the plain board: the reference for damage won back and cost '
               'without liars) and with enforceK 1-3; held out: 8 and '
               '32 agents, mistaken members and colluding liars at 4 of 16; arm 4 and the warm start over sequences of 6 boards at '
               '8, 16 and 32 (at 16 also with the most active quarter of contesters removed after board 3, back after board 5); '
               'the best arm in 64-agent squads joined by bridges (descriptive; needs bridged rollouts)',
    hypotheses=dict(
        H19_1='permTest(damage won back at 4 liars of 16: a19contest vs a19credit) > 0, one-sided exact, 8 + 8 runs',
        H19_2='permTest(slope over 0, 2, 4, 6, 8 liars of the waiting rate, the share of non-guards\' answers using a posted value '
              'that waited for a second sender: a19contest vs a19fixed) > 0, one-sided exact, 8 + 8 runs',
        H19_3='constraint: cost without liars, honest pooled share at 0 liars of the warm start without guards (plain board) - '
              'a19contest, <= 0.05 (hierarchical bootstrap interval; raw welfare beside it)',
        H19_4='signFlip(specialization at 16, a19open - warm) > 0 over 5 runs (the kill criterion; p >= 1/32, no Holm family); '
              'slope over log2 size 8-32, persistence and recovery reported with intervals',
        H19_5='secondary: misreporting minus the warm start\'s per arm (intervals); no arm below a third of the warm start\'s answers',
        H19_6='secondary: waiting rate at lifetime 15 minus 30 (a19contest), interval and sign-flip p'),
    families=dict(primary=['H19_1', 'H19_2'], constraint=['H19_3'], killCriterion=['H19_4'], secondary=['H19_5', 'H19_6']),
    alpha=0.05, nboot=2000,
    claims='claim (rl17_grid), TAKEOVER, one evaluation host per evaluation model; runs_p19/<arm>_s<seed>, eval_p19',
    haltedRuns='evaluated with the last checkpoint whose policy produced no step of the halting streak (evalAdapter19: a step\'s '
               'rollouts and logged KL come from the policy before its update), the warm start if that is step 0; reported')


def rl19Config(arm, seed, model, out, recipe, squads, warm=None, plan=RL19_PLAN, kind=None, **over):
    """A Phase 19 run: RL_PLAN v2 + BASE19 + the recipe + the arm, guards and squads in the world (`kind`, default plan['world']:
    'bridges': BRIDGE19's coupling and one bridge per pair, rl19_bridge.bridgeWorld19); with `warm` (WARM_ANSWERS for these squads,
    {guards: table}) the world carries the table for the arm's guards (arm 4: the guard-free one) and the run has the penalty
    (P19-3) and the answer halt, else neither (the speed check). Arm 4's records come from plan['records'] (P19-5).
    trainSeed = seed - 1; a checkpoint every step."""
    w19 = needWorld19(plan['world'] if kind is None else kind)
    a = dict(ARMS19[arm])
    g, rec = a.pop('guards'), a.pop('records', False)
    rec = plan['records'] if rec is None else rec
    if rec is None:
        raise ValueError(f'{arm}: records are set by P19-5 when RL19_PLAN is frozen')
    world = (bridgeWorld19 if w19 == 'bridges' else lambda w, N: w)(BASE19['world'] + ((('guards', g),) if g else ()) + (('squads', int(squads)),), BASE19['N'])
    if warm:
        t = dict(warm)[g]
        world += (('warmAnswers', tuple((tuple(k), float(v)) for k, v in sorted(t.items() if isinstance(t, dict) else t))),)
    pen = dict(sharePenalty=float(plan['decisions']['P19_3']['value']), halfLife=float(plan['decisions']['P19_2']['value'])) if warm \
        else dict(haltAnswers=None, halfLife=float(plan['decisions']['P19_2']['value']))
    return RL19Config(model, out, **{**RL_PLAN['rl'], **BASE19, **recipe, **a, 'records': bool(rec), 'world': world, **pen,
                                     'trainSeed': seed - 1, 'ckptEvery': 1, **over})


def speedConfig19(model, out, recipe, squads):
    "The speed check: arm 2, seed 1, 3 steps, a 4,096-token window, SPEED19's world, no penalty and no answer halt (WARM_ANSWERS unknown)."
    return rl19Config(SPEED19['arm'], SPEED19['seed'], model, out, recipe, squads, kind=SPEED19['world'], steps=SPEED19['steps'],
                      maxLen=SPEED19['maxLen'])


def pilotConfig19(squads, model, out, recipe, warm):
    "A pilot run: arm 2, seed 1, 10 steps, in squads of `squads`, PILOT19_PLAN's world; `warm`: WARM_ANSWERS for these squads."
    return rl19Config(PILOT19_PLAN['arm'], PILOT19_PLAN['seed'], model, out, recipe, squads, warm, kind=PILOT19_PLAN['world'],
                      steps=PILOT19_PLAN['steps'])


def pilotChoice19(gains):
    "The pilot's selection: {squad size: gain} -> the size with the larger gain; squads of 4 win ties."
    return max(sorted(gains), key=lambda S: (gains[S], S == 4))


def projectHours19(stepSeconds, steps=STEPS19, seeds=SEEDS19, pilotSeconds=None, evalHours=28.0, unitsPerHour=None):
    """A100 hours from seconds per step at 16 agents (each update training a fixed number of replies, so every arm costs the
    same per step): per run, per arm, training, the pilot (2 runs x 10 steps, plus about 0.5 hours of evaluation), the
    evaluation and the total; with unitsPerHour, also Colab compute units."""
    run = stepSeconds * steps / 3600
    arms = {a: n * run for a, n in seeds.items()}
    pilot = 2 * PILOT19_PLAN['steps'] * (pilotSeconds or stepSeconds) / 3600 + 0.5
    out = dict(perRun=run, arms=arms, training=sum(arms.values()), pilot=pilot, evaluation=float(evalHours))
    out['total'] = out['training'] + pilot + out['evaluation']
    return dict(out, units=out['total'] * unitsPerHour) if unitsPerHour else out


def evalAdapter19(runDir, haltAnswers=BASE19['haltAnswers'], haltRun=3, haltKLRun=FIX18['haltKLRun']):
    """The adapter a Phase 19 run is evaluated with: `final`, or for a halted run the last checkpoint whose policy produced no
    step of the halting streak (None, the warm start, if that is step 0). Step m's rollouts (answers, invalid replies) and its
    logged KL (one pass: measured before the update) come from ckpt-(m-1), the policy before step m's update, so a streak of
    n steps ending at step h began with ckpt-(h-n): the adapter is ckpt-(h-n-1). A jump halt at loop step s (halted.json
    'step' = s, two roll-backs in a row): ckpt-(s-1) also jumped when replayed, so the adapter is ckpt-(s-2).
    Unlike rl18_grid.evalAdapter (ckpt-(h-n)), which Phase 18 kept; none of its runs halted."""
    from pathlib import Path
    d = Path(runDir)
    if not (d / 'halted.json').exists():
        return str(d / 'final')
    h = json.loads((d / 'halted.json').read_text())
    why = h['reason']
    n = 1 if why.startswith('jump') else haltAnswers if 'answers' in why else haltRun if 'invalid' in why else haltKLRun
    k = h['step'] - n - 1
    return str(d / f'ckpt-{k:04d}') if k > 0 else None

## Tests

In [ ]:
# test_rl19_hash (FR-B1): Phase 17-18 hashes unchanged; an RL19Config at the defaults hashes as its RL17Config; each Phase 19 field
# changes the hash; the arms, runs, plans and projection
from dataclasses import asdict
from explRL.coop_grid import coopHash
from explRL.rl17_grid import PILOT17_PLAN, configHash17, rl17Config
from explRL.rl18_grid import rl18Config
from explRL.rl19_grid import *
D = '/content/drive/MyDrive/explRL_rl/'               # the Colab paths of the saved runs (their config.json hashes)
c17 = rl17Config('r17mix', 1, D + 'warm_model', D + 'runs_p17/r17mix_s1', RL17_PLAN['recipe'])
c18 = rl18Config('c18team', 2, D + 'warm_model', D + 'runs_p18/c18team_s2', RL17_PLAN['recipe'], 30, 3072)
for c in (c17, c18, RL17Config('m', 'o')):
    assert configHash19(c) == configHash17(c) and configHash19(RL19Config(**asdict(c))) == configHash17(c)
assert configHash19(c17) == '3a985454ee0c' and configHash19(c18) == '68da80473b25'   # runs_p17/r17mix_s1, runs_p18/c18team_s2
assert len({configHash19(RL19Config('m', 'o', **{k: v})) for k, v in dict(contestSlots='guards', halfLife=2.0, sharePenalty=0.5,
            threat=VARIED, records=True, haltAnswers=3).items()} | {configHash19(RL19Config('m', 'o'))}) == 7
assert len(RUNS19) == 29 and sum(SEEDS19.values()) == 29 and RUNS19[:4] == [(a, 1) for a in ARMS19]
assert BASE19['klClamp'] == 10.0 and BASE19['haltKLRun'] == 2 and BASE19['wrongPenalty'] == 2.5 and BASE19['maxLen'] == 3072
# the training world (P19-1): Gate C1 recorded (T5b), so squads joined by bridges, bridge1 on every arm (P19-8); squads
# without coupling only if Gate C1 had failed
assert SPEED19['world'] == PILOT19_PLAN['world'] == RL19_PLAN['world'] == DECISIONS19['P19_1']['value'] == worldFor19(RL19_PLAN['gates']['C1']) == 'bridges'
assert BRIDGES_READY and worldFor19(True) == 'bridges' and worldFor19(False) == 'squads' and needWorld19('squads') == 'squads' and needWorld19('bridges') == 'bridges'
for bad in (lambda: worldFor19(None), lambda: needWorld19('one board')):
    try:
        bad()
        raise AssertionError(bad)
    except ValueError:
        pass
br = rl19Config('a19contest', 1, 'm', 'o', {}, 4)
assert dict(br.world)['couple'] == 0.25 and dict(br.world)['bridges'] == 1 and 'bridgeGuard' not in dict(br.world)
assert dict(speedConfig19('m', 'o', {}, 8).world)['bridges'] == 1 and dict(pilotConfig19(4, 'm', 'o', {}, None).world)['couple'] == 0.25
PL = dict(RL19_PLAN, world='squads')              # the fallback world (had Gate C1 failed): squads without coupling
assert 'couple' not in dict(rl19Config('a19contest', 1, 'm', 'o', {}, 4, plan=PL).world) and configHash19(br) != configHash19(rl19Config('a19contest', 1, 'm', 'o', {}, 4, plan=PL))
tab = lambda v: [((16, n, L), v) for n in (0, 4, 6) for L in LIFETIMES]
warm = {4: tab(6.0), 0: tab(9.0)}                 # WARM_ANSWERS for one squad size: with 4 guards (arms 1-3) and without (arm 4)
cfg = rl19Config('a19contest', 3, 'm', 'o', RL17_PLAN['recipe'], 4, warm, PL)
w = dict(cfg.world)
assert cfg.trainSeed == 2 and cfg.scheme == 'contest' and cfg.contestSlots == 'guards' and cfg.sharePenalty == 0.5 and cfg.haltAnswers == 3
assert rl19Config('a19contest', 3, 'm', 'o', RL17_PLAN['recipe'], 4, {4: dict(tab(6.0))}, PL).world == cfg.world
assert w['guards'] == 4 and w['squads'] == 4 and w['colony'] and w['r'] == 2 and dict(w['warmAnswers'])[(16, 4, 15)] == 6.0
assert cfg.mixN == ((16, 1.0),) and cfg.threat == VARIED and rl19Config('a19fixed', 1, 'm', 'o', {}, 8, warm, PL).threat == FIXED4
SPEED19['world'] = 'squads'
try:
    sp = speedConfig19('m', 'o', RL17_PLAN['recipe'], 8)
finally:
    SPEED19['world'] = DECISIONS19['P19_1']['value']
assert sp.sharePenalty == 0 and sp.haltAnswers is None and sp.steps == 3 and sp.maxLen == 4096 and 'warmAnswers' not in dict(sp.world)
o4 = rl19Config('a19open', 1, 'm', 'o', {}, 4, warm, plan=dict(PL, records=True))
assert 'guards' not in dict(o4.world) and o4.records and dict(dict(o4.world)['warmAnswers'])[(16, 4, 15)] == 9.0   # arm 4: the guard-free table
try:
    rl19Config('a19open', 1, 'm', 'o', {}, 4, warm, PL)
    raise AssertionError('arm 4 without P19-5')
except ValueError:
    pass
assert configHash19(rl19Config('a19credit', 1, 'm', 'o', {}, 4, warm, PL)) != configHash19(rl19Config('a19contest', 1, 'm', 'o', {}, 4, warm, PL))
assert pilotChoice19({4: 0.1, 8: 0.2}) == 8 and pilotChoice19({4: 0.1, 8: 0.1}) == 4 and pilotChoice19({8: 0.3, 4: 0.1}) == 8
h = projectHours19(1080)                            # the README's 9 hours per run: 261 + 6.5 + 28
assert abs(h['perRun'] - 9) < 1e-9 and abs(h['training'] - 261) < 1e-9 and abs(h['total'] - 295.5) < 1e-9
assert abs(projectHours19(1080, unitsPerHour=11.8)['units'] - 295.5 * 11.8) < 1e-6
for p in (SPEED19, PILOT19_PLAN, RL19_PLAN):
    assert p['frozen'] is None and coopHash(p)                  # unfrozen until the user freezes it; JSON-hashable
assert sorted(k for k in RL19_PLAN['decisions'] if k.startswith('P19_')) == [f'P19_{i}' for i in range(1, 8)] and RL19_PLAN['decisions']['power']['value'] == 1.5 and RL19_PLAN['families']['primary'] == ['H19_1', 'H19_2']
assert RL19_PLAN['families']['secondary'] == ['H19_5', 'H19_6'] and 'exploratory' not in RL19_PLAN['families']
# a halted run's adapter: step m's rollouts and logged KL come from ckpt-(m-1), so a streak of n steps ending at h began with
# ckpt-(h-n) and the last policy outside it is ckpt-(h-n-1); a jump halt at loop step s: ckpt-(s-2); final without a halt
import json, tempfile
from pathlib import Path
from explRL.rl18_grid import evalAdapter
with tempfile.TemporaryDirectory() as d:
    assert evalAdapter19(d) == str(Path(d, 'final'))
    for step, why, k in ((7, "answers per board below a third of the warm start's for 3 steps (steps 5-7)", 3),
                         (7, 'mean KL 0.6 above 0.5 at step 7 (2 steps in a row)', 4), (43, 'invalid replies above 0.1 for 3 steps (steps 41-43)', 39),
                         (10, 'jump: invalid replies above 0.15 after two roll-backs in a row (step 11)', 8), (4, 'answers ... (steps 2-4)', None)):
        Path(d, 'halted.json').write_text(json.dumps(dict(step=step, reason=why)))
        assert evalAdapter19(d) == (None if k is None else str(Path(d, f'ckpt-{k:04d}'))), (why, evalAdapter19(d))
    Path(d, 'halted.json').write_text(json.dumps(dict(step=7, reason='mean KL 0.6 above 0.5 at step 7 (2 steps in a row)')))
    assert evalAdapter(d) == str(Path(d, 'ckpt-0005'))           # Phase 18's rule is left as it was (none of its runs halted)

In [ ]:
# AC-6: the Colab notebook's three Phase 19 cells compile, and each stops at its freeze assertion while its plan is unfrozen
# (RL19_PLAN also needs the user's confirmation); with SPEED19 frozen the speed cell's subprocess code compiles too
import json, tempfile
from pathlib import Path
import explRL
import explRL.rl19_grid as G
nb = json.load(open(Path(explRL.__file__).parents[1] / 'nbs' / 'colab_clue_rl.ipynb'))
src = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code' and 'rl19_grid import' in ''.join(c['source'])]
assert len(src) == 3 and all(s.startswith('#| gpu') for s in src)
for s, plan in zip(src, ('SPEED19', 'PILOT19_PLAN', 'RL19_PLAN')):
    compile(s, plan, 'exec')
    assert f"assert {plan}['frozen']" in s
    with tempfile.TemporaryDirectory() as d:
        try:
            exec(s, dict(OUT=Path(d), HOST='test', TAKEOVER=[], json=json, time=__import__('time'), stream=None, packZip=None))
            raise AssertionError(f'{plan}: ran unfrozen')
        except AssertionError as e:
            assert 'freeze' in str(e), e
assert "RL19_PLAN['userConfirmed']" in src[2] and 'WARM_ANSWERS' in src[1] and 'claim(' in src[1] and 'claim(' in src[2]
seen = []
def stub(code):
    compile(code, 'stream', 'exec')
    seen.append(code)
    return 1
# frozen, each cell then stops until Gate C1 is recorded and its world matches it (P19-1)
for s, plan in zip(src, (G.SPEED19, G.PILOT19_PLAN, G.RL19_PLAN)):
    assert "RL19_PLAN['gates']['C1']" in s and 'worldFor19(C1)' in s
plans = (G.SPEED19, G.PILOT19_PLAN, G.RL19_PLAN)
saved = [dict(p) for p in plans] + [G.WARM_ANSWERS]
try:
    for p in plans:
        p.update(frozen=True)
    G.RL19_PLAN.update(userConfirmed=True, squads=4, warmAnswers={4: {4: [], 0: []}}, records=False)
    G.WARM_ANSWERS = {4: {4: [], 0: []}}
    G.RL19_PLAN['gates'] = dict(C1=None, A2=None)             # unrecorded: every cell stops at its Gate C1 assertion
    for s in src:
        with tempfile.TemporaryDirectory() as d:
            try:
                exec(s, dict(OUT=Path(d), HOST='test', TAKEOVER=[], json=json, time=__import__('time'), stream=None, packZip=None))
                raise AssertionError('ran without Gate C1')
            except AssertionError as e:
                assert 'Gate C1' in str(e), e
    G.RL19_PLAN['gates'] = dict(C1=False, A2=None)            # Gate C1 failed: the fallback world, squads without bridges
    for p in plans:
        p['world'] = 'squads'
    with tempfile.TemporaryDirectory() as d:
        try:
            exec(src[0], dict(OUT=Path(d), HOST='test', TAKEOVER=[], json=json, time=__import__('time'), stream=stub, packZip=None))
        except RuntimeError as e:
            assert 'speed check' in str(e) and len(seen) == 1 and 'speedConfig19' in seen[0]
    G.RL19_PLAN['gates'] = dict(C1=G.GATE_C1, A2=None)        # T5b: Gate C1 as recorded, so bridges, and each cell passes its assertion
    for p in plans:
        p['world'] = 'bridges'
    seen.clear()
    with tempfile.TemporaryDirectory() as d:
        try:
            exec(src[0], dict(OUT=Path(d), HOST='test', TAKEOVER=[], json=json, time=__import__('time'), stream=stub, packZip=None))
        except RuntimeError as e:
            assert 'speed check' in str(e) and len(seen) == 1 and 'speedConfig19' in seen[0]
    # the main cell trains only the first 2 runs until STABLE19 is set (the stability check), then the rest
    import re
    trained = []

    def train(code):                                           # a stub run: its final adapter appears at once
        if 'trainRL19' not in code:
            return 1
        n = re.search(r"runs_p19/(\w+)'", code).group(1)
        trained.append(n)
        (Path(d) / 'runs_p19' / n / 'final').mkdir(parents=True, exist_ok=True)
        (Path(d) / 'runs_p19' / n / 'final' / 'adapter_model.safetensors').write_bytes(b'x')
        return 0
    with tempfile.TemporaryDirectory() as d:
        g = dict(OUT=Path(d), HOST='test', TAKEOVER=[], json=json, time=__import__('time'), stream=train, packZip=None)
        exec(src[2], g)
        exec(src[2], g)                                          # rerun: still waiting for the stability check
        assert trained == ['a19credit_s1', 'a19contest_s1'], trained
        g['STABLE19'] = True
        try:
            exec(src[2], g)
            raise AssertionError('evaluation ran on a stub')
        except RuntimeError as e:
            assert 'evaluation of' in str(e), e
        assert len(trained) == 29 and trained[2:4] == ['a19fixed_s1', 'a19open_s1']
finally:
    for p, v in zip(plans, saved):
        p.clear()
        p.update(v)
    G.WARM_ANSWERS = saved[-1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()